# Deduplicating and Geofencing final datasets

In [29]:
import pandas as pd
import geopandas as gpd
from sklearn.cluster import DBSCAN
from rapidfuzz import fuzz # for dropping fuzzy duplicates 

In [34]:
# Geofencing for Texas Local Food Directory Dataset
target_counties = gpd.read_file('texas_counties.geojson')

raw_points = pd.read_csv('tx_local_food_direct.csv')
raw_points['latitude'] = raw_points['latitude ']
raw_points.at[0, 'longitude'] = '-96.75077815570634'

# converting to GDF 
raw_points_df = gpd.GeoDataFrame(
    raw_points, 
    geometry=gpd.points_from_xy(raw_points['longitude'], raw_points['latitude']),
    crs="EPSG:4326" 
)

# 2. verifying that the CRS matches
if raw_points_df.crs != target_counties.crs:
    raw_points_df = raw_points_df.to_crs(target_counties.crs)

# 3. removing any points with empty geometry 
valid_points = raw_points_df[raw_points_df.geometry.is_valid & ~raw_points_df.geometry.is_empty]

# spatial join, using within to drop anything outside of the county polygons 
joined_localfood = gpd.sjoin(valid_points, target_counties, how="inner", predicate="within")

In [31]:
raw_points.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   farm_id         27 non-null     object 
 1   farm_name       27 non-null     object 
 2   street_address  27 non-null     object 
 3   city            27 non-null     object 
 4   county          27 non-null     object 
 5   zip_code        27 non-null     int64  
 6   latitude        27 non-null     float64
 7   longitude       27 non-null     object 
 8   profile_url     27 non-null     object 
 9   date_collected  27 non-null     object 
 10  latitude        27 non-null     float64
dtypes: float64(2), int64(1), object(8)
memory usage: 2.4+ KB


*Did not remove any points*

In [37]:
# Geofencing for miscellaneous sources dataset
target_counties = gpd.read_file('texas_counties.geojson')

raw_points = pd.read_csv('farmers_market_local_list.csv')

# converting to GDF 
raw_points_df = gpd.GeoDataFrame(
    raw_points, 
    geometry=gpd.points_from_xy(raw_points['longitude'], raw_points['latitude']),
    crs="EPSG:4326" 
)

2. verifying that the CRS matches
if raw_points_df.crs != target_counties.crs:
    raw_points_df = raw_points_df.to_crs(target_counties.crs)

# 3. removing any points with empty geometry 
valid_points = raw_points_df[raw_points_df.geometry.is_valid & ~raw_points_df.geometry.is_empty]

# spatial join, using within to drop anything outside of the county polygons 
joined_misc = gpd.sjoin(valid_points, target_counties, how="inner", predicate="within")

In [28]:
joined_misc.to_csv('farmers_market_local_list.csv', index=False)

Removed one farm

Index(['farm_name', 'street_address', 'city', 'county', 'zip_code', 'latitude',
       'longitude', 'website', 'date_collected', 'geometry', 'index_right',
       'MTFCC', 'OID', 'GEOID', 'STATE', 'COUNTY', 'COUNTYNS', 'BASENAME',
       'NAME', 'LSADC', 'FUNCSTAT', 'COUNTYCC', 'AREALAND', 'AREAWATER',
       'OBJECTID', 'CENTLAT', 'CENTLON', 'INTPTLAT', 'INTPTLON'],
      dtype='object')